# Fig 3B / S6A — Pairwise similarity of LD references (data processing)

Mean per-region MSE, max |ΔLD| and concordance between every pair of LD references (in-sample GTEx, ADSP sketch, UKB sketch, 1000 Genomes).

**Outputs:**
- `figure_data/Fig3B.LD_Pairwise_Similarity.rds` — GTEx (all ancestries) as in-sample, matching the Fig 3A simulation.
- `figure_data/FigS6A.LD_Pairwise_Similarity_GTEx_EUR.rds` — same with GTEx EUR as in-sample, matching the Fig S6B simulation.

In [1]:
source("../../R/utils.R")

## aggregate pairwise metrics over regions -> symmetric long table (diagonal = NA)
pairwise_heat_data <- function(pairwise_res, ref_lab,
                               metrics = c("mse", "max_abs_diff", "concord")) {
  ref_lv <- unname(ref_lab)
  agg <- pairwise_res %>%
    separate(comparison, c("ref_a", "ref_b"), sep = "_vs_", remove = FALSE) %>%
    group_by(ref_a, ref_b) %>%
    summarize(across(all_of(metrics), ~ mean(.x, na.rm = TRUE)), .groups = "drop")

  agg %>%
    pivot_longer(all_of(metrics), names_to = "metric", values_to = "val") %>%
    { bind_rows(., rename(., ref_a = ref_b, ref_b = ref_a)) } %>%   # symmetrize
    mutate(
      x = factor(ref_lab[ref_a], levels = ref_lv),
      y = factor(ref_lab[ref_b], levels = rev(ref_lv)),
      metric = factor(metric, levels = metrics)
    ) %>%
    complete(metric, x, y)                                          # diagonal -> NA (blank)
}

## Fig 3B: GTEx, ADSP, UKB, 1000 Genomes

In [ ]:
pairwise_res <- readRDS("/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/LD_pairwise/ld_pairwise_combined.rds")

plot_dt <- pairwise_heat_data(pairwise_res,
                              ref_lab = c(InSample = "GTEx", ADSP = "ADSP", UKB = "UKB", KG = "1000G"))
save_data(plot_dt, "Fig3B.LD_Pairwise_Similarity")

## Fig S6A: GTEx EUR as in-sample

In [ ]:
pairwise_res_eur <- readRDS("/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/LD_pairwise_EUR///ld_pairwise_combined.rds")

plot_dt_eur <- pairwise_heat_data(pairwise_res_eur,
                                  ref_lab = c(InSample = "GTEx EUR", ADSP = "ADSP", UKB = "UKB", KG = "1000G"))
save_data(plot_dt_eur, "FigS6A.LD_Pairwise_Similarity_GTEx_EUR")